In [0]:
# Master Import Cell
from pyspark.sql.functions import col, when, udf, length
from pyspark.sql.types import FloatType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

# Confirming imports are loaded
print("All Spark functions and ML libraries imported successfully.")


All Spark functions and ML libraries imported successfully.


In [0]:
# Install NLP library
%pip install textblob

# Restart Python to register the new library
dbutils.library.restartPython()

# Standard Imports
from pyspark.sql.functions import col, length, when, udf
from pyspark.sql.types import FloatType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from textblob import TextBlob


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql.functions import col, length, get_json_object

# Defining Production Paths
reviews_path = "/Volumes/workspace/default/project/Toys_and_Games.jsonl.gz"
meta_path = "/Volumes/workspace/default/project/meta_Toys_and_Games.jsonl.gz"

# 2. Ingesting Reviews
df_reviews = spark.read.json(reviews_path)

# 3. Robust Metadata Ingestion (To Handle the duplicate column error)
df_meta_raw = spark.read.text(meta_path)

df_meta = df_meta_raw.select(
    get_json_object(col("value"), "$.parent_asin").alias("parent_asin"),
    get_json_object(col("value"), "$.main_category").alias("meta_main_category"),
    get_json_object(col("value"), "$.price").alias("meta_price"),
    get_json_object(col("value"), "$.title").alias("meta_title")
).dropna(subset=["parent_asin"])

# 4. Data Profile: Record Counts
print(f"Total Review Records: {df_reviews.count()}")
print(f"Total Metadata Records: {df_meta.count()}")

# 5. Exploratory Data Analysis
df_reviews = df_reviews.withColumn("review_length", length(col("text")))

print("--- Rating Distribution ---")
display(df_reviews.groupBy("rating").count().orderBy("rating"))

print("--- Review Length Distribution ---")
display(df_reviews.select("review_length"))




Total Review Records: 16260406
Total Metadata Records: 890874
--- Rating Distribution ---


rating,count
1.0,1631131
2.0,767908
3.0,1103648
4.0,1797010
5.0,10960709


--- Review Length Distribution ---


review_length
635
510
43
387
443
238
44
144
332
74


In [0]:
from pyspark.sql.functions import col, when

# Distributed Join
# We used an inner join on 'parent_asin' to ensure we only keep reviews with matching product info
df_joined = df_reviews.join(df_meta, on="parent_asin", how="inner")

# Data Cleaning
# Standardizing 'price' to a numeric format and defining our target 'label'
# A rating > 4 is defined as a 'Success' (1), otherwise (0)
df_prepared = df_joined.withColumn("price", col("meta_price").cast("float")) \
                       .withColumn("label", when(col("rating") > 4, 1).otherwise(0))

# Handling Missing Values
# Removing any rows that lack essential data to prevent errors during modeling
df_clean = df_prepared.dropna(subset=["price", "text", "meta_main_category", "rating"])

print("Phase 3: Data Preparation Complete.")
print(f"Final Record Count for Modeling: {df_clean.count()}")

Phase 3: Data Preparation Complete.
Final Record Count for Modeling: 11230690


In [0]:
from textblob import TextBlob
from pyspark.sql.functions import udf, col
from pyspark.sql.types import FloatType

# Defining Sentiment Logic
def get_sentiment(text):
    if text is None:
        return 0.0
    return TextBlob(text).sentiment.polarity

# Registering the User Defined Function (UDF)
sentiment_udf = udf(get_sentiment, FloatType())

# Applying Sentiment Analysis to the 11.2M records
df_analytics = df_clean.withColumn("sentiment_score", sentiment_udf(col("text")))

# Statistical Validation: Pearson Correlation
correlation = df_analytics.stat.corr("sentiment_score", "rating")

print(f"Phase 4: Analytical Depth Complete.")
print(f"Pearson Correlation (Sentiment vs Rating): {correlation:.4f}")

# Preview results
display(df_analytics.select("text", "rating", "sentiment_score").limit(10))

Phase 4: Analytical Depth Complete.
Pearson Correlation (Sentiment vs Rating): 0.4604


text,rating,sentiment_score
"I purchased several of these for my granddaughters for Xmas & they absolutely love them. In fact so do their brothers. Lol. Be aware that there are tiny magnets in them so that you can perch the ppl onto the animals & keep them there. I almost didn’t buy them bc of that given all the news stories about kids and magnets, but the magnets are INSIDE the toys. So as long as ur kids aren’t eating their toys you should be okay, but be careful around your pets so that the pets don’t try to eat them. My grandkids adore all things Schleich as do I bc I bought it for their parents too decades ago. Schleich is well made & lasts forever.",5.0,0.2375
"It’s cute, but it arrived broken & has some pretty sharp points on it. Also, it has gaps little fingers can get trapped in while in motion. The plastic is extremely hard & not pliable at all. Not sure yet if I will order a replacement or not. It’s for an older child, but he has younger siblings. It’s cute. I haven’t got it running yet bc it doesn’t come with batteries so that’s next. As it arrived broken it’s definitely going back. It could be an awesome product if made with better materials though.",3.0,0.0625
This is very cute but beware that the animals are only painted on one side of the figure. I used for my daughters circus birthday party and loved that it was a bit more on the girly side. I also purchased the Janod brand and combined the two circus trains -worked great together. Only complaint is that they really should have both sides of the animal figures painted.... Not unhappy with my purchase but not sure I'd buy again if I knew that.,3.0,0.26
I love the laugh and learn stuff do I wanted to love this but its just OK. The electronic element makes the bag a bit heavy even when empty. My daughter uses but not a ton and usually gravitate to another bag if there are others available,3.0,0.20625
Part of my childhood reborn and newly gifted to my kids! It brings me such joy to have my kids play with snap bracelets! they are soft on one side and colorful sequins on the other side! they change color when you wipe the surface. HUGE hit for the kids! They are all different colors and they slap onto your wrist. =),5.0,0.31875
soft. cute. what more do you need? also classy. its not all gaudy like some animals.,5.0,0.3
super cute. my 4 year old loves it,5.0,0.31111112
"I ordered this for our family Christmas celebration & everyone LOVED it! It super easy & fun! We got lots of laughs! I just ordered ""The Office: Expansion Pack"" as all our kids are big Office fans! Can't wait to try that too & add to the fun! Would highly recommend! A+",5.0,0.38363096
love it,4.0,0.5
Helped my little one enjoy bath time again!,5.0,0.15625


In [0]:
# Product Performance Analysis

from pyspark.sql.functions import avg, count

performance_analysis = df_clean.groupBy("meta_main_category").agg(
    avg("helpful_vote").alias("avg_helpfulness"),
    avg("rating").alias("avg_rating"),
    count("rating").alias("review_count")
).orderBy("avg_helpfulness", ascending=False)

print("Phase 4.5: Product Performance Analysis Complete.")
# Showing the top categories where reviews are most influential
display(performance_analysis)

Phase 4.5: Product Performance Analysis Complete.


meta_main_category,avg_helpfulness,avg_rating,review_count
Premium Beauty,2.75,4.027777777777778,36
Sports Collectibles,2.509433962264151,3.6792452830188678,53
Software,2.4879227053140096,2.6666666666666665,207
Camera & Photo,1.970937939754859,4.032835029469949,49033
GPS & Navigation,1.6304347826086956,3.608695652173913,46
Portable Audio & Accessories,1.5254237288135593,4.288135593220339,59
Collectible Coins,1.456140350877193,4.385964912280702,57
Video Games,1.3189545261368465,4.226413089672758,9412
Cell Phones & Accessories,1.3171226646140928,3.9948219158911202,14291
Home Audio & Theater,1.289107289107289,3.9393939393939394,2442


In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# Encode Categorical Data
indexer = StringIndexer(inputCol="meta_main_category", outputCol="category_index", handleInvalid="skip")
encoder = OneHotEncoder(inputCol="category_index", outputCol="category_vec")

#Vector Assembly
assembler = VectorAssembler(
    inputCols=["sentiment_score", "price", "category_vec"], 
    outputCol="features"
)

# Transform the Data
df_indexed = indexer.fit(df_analytics).transform(df_analytics)
df_encoded = encoder.fit(df_indexed).transform(df_indexed)
ml_data = assembler.transform(df_encoded)

print("Phase 5: Feature Engineering Complete.")
# Preview the features vector
display(ml_data.select("features", "label").limit(5))

Phase 5: Feature Engineering Complete.


features,label
"{""type"":""0"",""size"":""34"",""indices"":[""0"",""1"",""2""],""values"":[""0.3816964328289032"",""29.3700008392334"",""1.0""]}",1
"{""type"":""0"",""size"":""34"",""indices"":[""0"",""1"",""2""],""values"":[""0.32499998807907104"",""29.3700008392334"",""1.0""]}",0
"{""type"":""0"",""size"":""34"",""indices"":[""0"",""1"",""2""],""values"":[""0.25"",""29.3700008392334"",""1.0""]}",0
"{""type"":""0"",""size"":""34"",""indices"":[""0"",""1"",""2""],""values"":[""-0.1944444477558136"",""29.3700008392334"",""1.0""]}",0
"{""type"":""0"",""size"":""34"",""indices"":[""0"",""1"",""2""],""values"":[""0.39615383744239807"",""29.3700008392334"",""1.0""]}",1


In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Train/Test Split
train_data, test_data = ml_data.randomSplit([0.7, 0.3], seed=42)

# Model Training
# Initializing the Logistic Regression 
lr = LogisticRegression(featuresCol="features", labelCol="label")
lr_model = lr.fit(train_data)

# Predictions
predictions = lr_model.transform(test_data)

# Evaluation: F1-Score
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
f1_score = evaluator.evaluate(predictions)

print(f"Final Project Evaluation - F1-Score: {f1_score:.4f}")

# Confusion Matrix
print("--- Confusion Matrix (Actual vs. Predicted) ---")
predictions.crosstab("prediction", "label").show()

Final Project Evaluation - F1-Score: 0.7042
--- Confusion Matrix (Actual vs. Predicted) ---
+----------------+------+-------+
|prediction_label|     0|      1|
+----------------+------+-------+
|             0.0|438391| 323006|
|             1.0|624662|1984047|
+----------------+------+-------+



In [0]:
# Phase 7: Deployment - Clean Export
export_path = "/Volumes/workspace/default/project/toy_analysis_results.csv"

# Sampling 10,000 rows
df_export = predictions.select(
    "text", 
    "rating", 
    "sentiment_score", 
    "prediction", 
    "label", 
    "meta_main_category"
).limit(10000).toPandas()

df_export.to_csv(export_path, index=False)
print(f"Deployment successful! Exported to: {export_path}")

Deployment successful! Exported to: /Volumes/workspace/default/project/toy_analysis_results.csv
